# Secure Multi-party Computation: Private Set Intersection 
Secure Multi-party Computation (SMPC) is a subfield of cryptography that enables multiple parties
to jointly compute a function over their inputs while keeping those inputs private.
The goal is to ensure that no party learns anything more than what can be inferred from their own
input and the output of the computation.

This work focuses on a specific SMPC operation named Private Set Intersection (PSI).
PSI is a cryptographic protocol that allows two parties to compute the intersection of their
private sets without revealing any other information about the sets. Some of the existing
implementations of PSI, their advantages and disadvantages are presented in this document.

## Preparation of Sets

In [1]:
def load_set(path: str) -> set[str]:
    with open(path, "r") as f:
        return set(line.strip() for line in f)

# Load sets with 1000 random strings
# Sets were created using command: sort -R rockyou2000.txt | head -n 1000 > party_X.txt
party_a_set = load_set("data/party_a.txt")
party_b_set = load_set("data/party_b.txt")

intersection_size = len(party_a_set.intersection(party_b_set))
print(f"Intersection size: {intersection_size}")

Intersection size: 507


## Bloom Filter-based PSI

In [2]:
from src.psi_bloom_filter import Bloom, BloomFilterPSI
import time

# Initialize Bloom filters
expected_items = 1000
false_positive_rate = 0.01
party_a_bloom = Bloom(expected_items, false_positive_rate)
party_b_bloom = Bloom(expected_items, false_positive_rate)

# Add items to Bloom filters
start_time = time.time()
for item_a in party_a_set:
    party_a_bloom.add(item_a)

for item_b in party_b_set:
    party_b_bloom.add(item_b)
end_time = time.time()
print(f"Bloom filter construction time: {end_time - start_time} seconds")

# Compute intersection
start_time = time.time()
psi_bloom = BloomFilterPSI([party_a_bloom, party_b_bloom]).intersection()
end_time = time.time()
bloom_time = end_time - start_time
print(f"Bloom filter PSI computation time: {bloom_time} seconds")

# Get PSI elements from party A point of view
start_time = time.time()
psi_element_count = sum(1 for item in party_a_set if item in psi_bloom)
end_time = time.time()
psi_time = end_time - start_time
print(f"A: PSI item evaluation time: {psi_time} seconds")
print(f"A: PSI element count: {psi_element_count}")
false_positives = psi_element_count - intersection_size
true_negatives = len(party_a_set) - psi_element_count
print(f"A: False positive rate: {false_positives / (false_positives + true_negatives) * 100}%")

# Get PSI elements from party B point of view
start_time = time.time()
psi_element_count = sum(1 for item in party_b_set if item in psi_bloom)
end_time = time.time()
psi_time = end_time - start_time
print(f"B: PSI item evaluation time: {psi_time} seconds")
print(f"B: PSI element count: {psi_element_count}")
false_positives = psi_element_count - intersection_size
true_negatives = len(party_b_set) - psi_element_count
print(f"B: False positive rate: {false_positives / (false_positives + true_negatives) * 100}%")


Bloom filter construction time: 0.0181581974029541 seconds
Bloom filter PSI computation time: 4.982948303222656e-05 seconds
A: PSI item evaluation time: 0.006628751754760742 seconds
A: PSI element count: 508
A: False positive rate: 0.2028397565922921%
B: PSI item evaluation time: 0.006224155426025391 seconds
B: PSI element count: 511
B: False positive rate: 0.8113590263691683%


## Diffie-Hellman-based PSI

In [3]:
from src.psi_diffie_hellman import DH_PSI_Party

# Safe prime for DH from RFC 3526 MODP Group #14
p = int(
    "".join(
        """FFFFFFFF FFFFFFFF C90FDAA2 2168C234 C4C6628B 80DC1CD1
        29024E08 8A67CC74 020BBEA6 3B139B22 514A0879 8E3404DD
        EF9519B3 CD3A431B 302B0A6D F25F1437 4FE1356D 6D51C245
        E485B576 625E7EC6 F44C42E9 A637ED6B 0BFF5CB6 F406B7ED
        EE386BFB 5A899FA5 AE9F2411 7C4B1FE6 49286651 ECE45B3D
        C2007CB8 A163BF05 98DA4836 1C55D39A 69163FA8 FD24CF5F
        83655D23 DCA3AD96 1C62F356 208552BB 9ED52907 7096966D
        670C354E 4ABC9804 F1746C08 CA18217C 32905E46 2E36CE3B
        E39E772C 180E8603 9B2783A2 EC07A28F B5C55DF0 6F4C52C9
        DE2BCBF6 95581718 3995497C EA956AE5 15D22618 98FA0510
        15728E5A 8AACAA68 FFFFFFFF FFFFFFFF""".split()
    ),
    16
)
# Setup private keys
start_time = time.time()
party_a = DH_PSI_Party(p)
party_b = DH_PSI_Party(p)
party_a_list = list(party_a_set)
party_b_list = list(party_b_set)
end_time = time.time()
print(f"Diffie-Hellman PSI setup time: {end_time - start_time} seconds")

# Compute intersection
start_time = time.time()
# A encrypts its items (H(item_a_i)^private_key_a)
# B encrypts its items (H(item_b_j)^private_key_b)
party_a_encrypted_items = party_a.prepare_items(party_a_set)
party_b_encrypted_items = party_b.prepare_items(party_b_set)

# A encrypts B's encrypted items (H(item_b_j)^(private_key_b)^private_key_a)
# B encrypts A's encrypted items (H(item_a_i)^(private_key_a)^private_key_b)
party_b_double_encrypted_items = party_a.process_received_items(party_b_encrypted_items)
party_a_double_encrypted_items = party_b.process_received_items(party_a_encrypted_items)

intersection = party_a.intersection(party_a_list, party_a_double_encrypted_items, party_b_double_encrypted_items)
# b_side_intersection = party_b.intersection(party_b_list, party_b_double_encrypted_items, party_a_double_encrypted_items)
end_time = time.time()
psi_time = end_time - start_time
print(f"PSI computation time: {psi_time} seconds")
print(f"PSI element count: {len(intersection)}")


Diffie-Hellman PSI setup time: 0.0002219676971435547 seconds
PSI computation time: 102.40228700637817 seconds
PSI element count: 507


In [16]:
print(len(party_a_set.intersection(party_b_set).union(set(intersection))) == intersection_size)

True
